# Exploratory Data Analysis: Steam's Highest-Revenue Games of 2024

**Holon Institute of Technology (HIT) — Faculty of Computer Science**

| | |
|---|---|
| **Course** | Introduction to Data Science |
| **Assignment** | 1 — Exploratory Data Analysis (EDA) |
| **Student name** | *[TO BE COMPLETED]* |
| **ID number** | *[TO BE COMPLETED]* |
| **Lecturer** | Dr. Ori Itai |
| **Teaching assistant** | Hanit Ohayon Hadad |
| **Submission** | Individual |
| **Due date** | 02.08.2026 |

---

## Table of Contents

1. [Introduction and Objective](#1-introduction-and-objective)
2. [Dataset Selection](#2-dataset-selection)
3. [Meta-Analysis of the Data](#3-meta-analysis-of-the-data)
4. [Data Quality and Completeness](#4-data-quality-and-completeness)
5. [Univariate Analysis](#5-univariate-analysis)
6. [Correlations and Relationships](#6-correlations-and-relationships)
7. [Index Analysis](#7-index-analysis)
8. [Insights and the Data Story](#8-insights-and-the-data-story)
9. [Extensions](#9-extensions)

## 1. Introduction and Objective

This notebook is an exploratory analysis of the 1,500 highest-revenue games
released on Steam during 2024.

The objective is not to produce charts, but to understand the dataset as a
**system**: its internal structure, the assumptions hidden inside it, and the
limitations and biases those assumptions impose on any conclusion drawn from
it. The analysis is guided throughout by a single principle:

> Data is not "truth". It is the output of a measurement process.

Each section therefore asks not only *what the numbers say*, but also **what was
measured, what is missing, what is biased, and what has been artificially
centred**. Where a finding contradicts what one would intuitively expect, the
contradiction is stated explicitly rather than smoothed over.

## 2. Dataset Selection

### 2.1 Verification of the Selection Requirements

The assignment defines four criteria a chosen dataset must satisfy: it must be
tabular, contain at least 1,000 rows and at least 10 columns, and combine
numeric, temporal and categorical variables.

These criteria are verified below rather than asserted, because the same checks
also produce the first structural facts about the data. The semantic role of
each column — numeric, temporal, categorical or identifier — is declared once in
`src/data_loading.py` and reused for the rest of the notebook, so that the
analysis never depends on a column list retyped in several places.

In [1]:
import sys
from pathlib import Path

import pandas as pd

# This notebook lives in <project>/notebooks while the helper modules live in
# <project>/src. Adding the project directory to the import path lets the
# notebook import them regardless of the directory Jupyter was started from.
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from src.data_loading import (
    RAW_DATA_PATH,
    build_column_inventory,
    load_raw_dataset,
    verify_selection_requirements,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

In [2]:
steam_games = load_raw_dataset()

row_count, column_count = steam_games.shape
print(f"Source file : {RAW_DATA_PATH.name}")
print(f"Rows        : {row_count:,}")
print(f"Columns     : {column_count}")

steam_games.head()

Source file : Steam_2024_bestRevenue_1500.csv
Rows        : 1,500
Columns     : 11


,name,releaseDate,copiesSold,price,revenue,avgPlaytime,reviewScore,publisherClass,publishers,developers,steamId
0,WWE 2K24,07-03-2024,165301,99.99,8055097.0,42.365140,71,AAA,2K,Visual Concepts,2315690
1,EARTH DEFENSE FORCE 6,25-07-2024,159806,59.99,7882151.0,29.651061,57,Indie,D3PUBLISHER,SANDLOT,2291060
2,Sins of a Solar Empire II,15-08-2024,214192,49.99,7815247.0,12.452593,88,Indie,Stardock Entertainment,"Ironclad Games Corporation,Stardock Entertainment",1575940
3,Legend of Mortal,14-06-2024,440998,19.99,7756399.0,24.797817,76,Indie,"Paras Games,Obb Studio Inc.",Obb Studio Inc.,1859910
4,Shin Megami Tensei V: Vengeance,13-06-2024,141306,59.99,7629252.0,34.258496,96,AA,SEGA,ATLUS,1875830


In [3]:
verify_selection_requirements(steam_games)

,Requirement,Required,Observed,Status
0,Tabular format,Yes,Yes (single flat CSV table),PASS
1,Minimum rows,>= 1000,1500,PASS
2,Minimum columns,>= 10,11,PASS
3,Numeric variables,>= 1,"5 (copiesSold, price, revenue, avgPlaytime, re...",PASS
4,Temporal variables,>= 1,1 (releaseDate),PASS
5,Categorical variables,>= 1,"3 (publisherClass, publishers, developers)",PASS


In [4]:
build_column_inventory(steam_games)

,Column,Assigned role,Stored dtype,Non-null,Missing,Unique values,Example value
0,name,identifier,str,1500,0,1500,WWE 2K24
1,releaseDate,temporal,str,1500,0,235,07-03-2024
2,copiesSold,numeric,int64,1500,0,1460,165301
3,price,numeric,float64,1500,0,58,99.99
4,revenue,numeric,float64,1500,0,1497,8055097.0
5,avgPlaytime,numeric,float64,1500,0,1500,42.36514
6,reviewScore,numeric,int64,1500,0,72,71
7,publisherClass,categorical,str,1500,0,4,AAA
8,publishers,categorical,str,1499,1,1131,2K
9,developers,categorical,str,1498,2,1406,Visual Concepts


#### What the verification shows

The dataset satisfies every selection criterion, but the two tables above
already raise several points that shape the rest of the analysis.

**The column margin is narrow.** The dataset has 11 columns against a required
minimum of 10. Only one column can be removed before the dataset stops meeting
the specification. This is a real constraint on later sections: a column that
turns out to carry little information cannot simply be dropped, it has to be
retained and its weakness documented instead.

**Two of the eleven columns are identifiers, not variables.** `name` and
`steamId` each hold 1,500 unique values across 1,500 rows. They carry no
distributional signal and will not appear in any statistical summary, but they
define what a row *is*, which makes them central to the duplicate analysis in
section 4 and the index analysis in section 7. Excluding them, only nine columns
are analytically substantive.

**Missing data is almost absent.** Three cells out of 16,500 are empty — around
0.02% — and all three sit in `publishers` and `developers`. A dataset this
complete is itself worth questioning: near-perfect completeness usually
indicates either a curated extract or fields that were imputed upstream rather
than genuinely observed. Section 4 examines which of these applies.

**`avgPlaytime` has 1,500 unique values in 1,500 rows.** A recorded average
playtime across thousands of players would be expected to produce at least some
repeated values, especially at the low end. A column that is unique for every
single row, carried to roughly fourteen decimal places, has the signature of a
*computed* quantity rather than a *recorded* one. This is the first concrete
hint that parts of this dataset are modelled rather than measured — a thread
picked up in section 2.2 and quantified in section 6.

**`releaseDate` is stored as text, not as a date.** The values follow a
`DD-MM-YYYY` layout, which pandas does not recognise automatically. Parsing it
requires an explicit format: relying on inference would silently misread every
date where both day and month are 12 or below, for example `01-02-2024`.
Section 3 performs this conversion deliberately for that reason.

### 2.2 Description of the Data Source

#### Origin of the data

The file was downloaded from **Kaggle** as a static CSV snapshot. No datasheet,
licence file, methodology note or version tag accompanied the download.

> **To be completed before submission:** the exact Kaggle dataset URL and its
> stated licence.

#### The underlying source, and why the numbers cannot be measurements

Valve does not publish per-title sales or revenue figures for Steam. No public
interface exposes how many copies a given game has sold. It follows that the
`revenue` and `copiesSold` columns **cannot be direct measurements**; they must
be estimates produced by a third-party analytics provider.

The established industry approach is to infer the number of owners from the
publicly visible review count using a review-to-owner multiplier — widely known
as the *Boxleiter method* — and then combine that estimate with observed pricing
to obtain revenue. Two pieces of internal evidence are consistent with the file
having been produced this way:

1. **`avgPlaytime` is unique for all 1,500 rows**, carried to roughly fourteen
   decimal places. Recorded averages do not behave like this; derived model
   outputs do.
2. **`revenue` is close to, but not identical to, `copiesSold` × `price`.** If
   revenue were a simple product of the two, the relationship would be exact. It
   is not, and a subset of rows even exceeds that product — which is what one
   would expect from a model that accounts for discount and regional price
   history rather than applying a single current price. Section 6 quantifies
   this relationship.

This distinction matters more than it may appear. It means the dataset does not
record what happened on Steam in 2024; it records **what one estimation model
believes happened**, and every downstream conclusion inherits that model's
assumptions and its unstated error.

#### Purpose of collection

No stated purpose ships with the file, so the purpose must be inferred from its
shape. The selected fields — revenue, copies sold, price, publisher class — are
precisely the fields required for commercial market intelligence, and the
ranking-and-truncation to a "top 1,500" is a market-report convention rather
than a sampling design. A dataset assembled for academic research would more
plausibly retain genre, tags, supported platforms, languages and raw review
counts, and would not discard everything below a revenue threshold.

The evidence therefore points to **commercial rather than scientific**
collection. Section 3 develops this distinction and its consequences in full.

#### The collecting entity

This is only partially answerable. The immediate distributor is the Kaggle
uploader; the upstream producer is an analytics vendor that the file does not
name. Which vendor, and which version of its model, cannot be determined from
the data alone. **The collecting entity is therefore effectively unknown**, and
this is recorded as a limitation rather than presented as resolved.

#### Domain context

Steam is the dominant storefront for PC games, which makes a top-revenue Steam
extract a reasonable proxy for the commercial upper tier of PC gaming — but only
for that tier. Steam receives many thousands of new titles per year, so 1,500
titles represent a small fraction of 2024 releases, and by construction the most
commercially successful fraction.

> **To be completed before submission:** cite a source for the total number of
> Steam releases in 2024, so the coverage fraction can be stated precisely
> rather than qualitatively.

One further ambiguity is material to any financial reading: revenue of this kind
is normally quoted **gross**, before Valve's platform cut, refunds and taxes.
The file does not state whether that is the case here. The difference between
gross and net is on the order of 30%, which is larger than most of the effects
this analysis will detect.

#### What the source does not tell us

The assignment notes that a lack of information is itself an insight. Here the
gaps are substantial, and each one is a question the data cannot answer:

| Unanswered question | Why it matters |
|---|---|
| On what date was the snapshot taken? | Revenue accumulates over time; without a snapshot date, the exposure window of each title is unknown. |
| Is revenue gross or net? | A roughly 30% difference in every monetary figure. |
| Which currency, and how were regions converted? | Affects comparability across titles. |
| Is `price` the launch price, the current price, or an average? | Determines whether price and revenue are even mutually consistent. |
| Is revenue lifetime-to-date or 2024-only? | Changes the meaning of every comparison against release date. |
| What is the estimation method, and what is its error? | No confidence interval can be reconstructed. |
| Under what licence is the data released? | Governs whether the analysis may be published. |

The practical consequence is a boundary on what this notebook may legitimately
claim: the figures support **relative** statements — comparisons, rankings,
associations between variables — but they do not support **absolute** financial
claims about any individual title.

#### The selection principle, and its consequence

The filename encodes the sampling rule: `bestRevenue_1500`. The rows were
selected *because* they earned the most. This is a truncated, ranked, top-of-
distribution sample, and it is the single most important fact about the dataset.

Every distributional statement in this notebook therefore describes **the
top 1,500 earning games of 2024**, never "games on Steam". A statement such as
"the average game earned 2.6 million dollars" would be false by construction:
the sample was selected on the very quantity being averaged. This selection
effect is revisited as a formal bias in section 8.

#### Motivation for choosing this dataset

> *[TO BE PERSONALISED]* — the assignment asks for a topic close to the
> student's own interests. Replace this paragraph with your own reason for
> choosing the Steam games market.

## 3. Meta-Analysis of the Data

*Not yet written.* This section will cover:

- **3.1 File analysis** — file size, format, creation date, purpose of the data.
- **3.2 Data structure** — row and column counts, column naming conventions, data types and what they reveal.
- **Discussion** — were the data collected for research or for operations, and which biases follow from the answer?

## 4. Data Quality and Completeness

*Not yet written.* This section will cover:

- **4.1 Missing data** — extent, pattern (random or structured), what the absence itself teaches, and how it could be imputed.
- **4.2 Duplicates** — full and partial duplicates, what they imply, and whether removal is the right response.
- **4.3 Suspicious values** — impossible values, placeholder values, and implausible zeros or negatives.
- **4.4 Cardinality** — columns without variance, and columns with an excess of unique values.

## 5. Univariate Analysis

*Not yet written.* This section will cover:

- **5.1 Numeric variables** — mean, median, standard deviation, MAD, minimum, maximum, quantiles and IQR; skew; distribution shape; outlier detection using three methods.
- **5.2 Categorical variables** — frequencies, the mode and its share, top-K coverage, the smallest set of values covering P% of the data, and the treatment of rare categories.
- **Discussion** — do the measures of central tendency represent this data well?

## 6. Correlations and Relationships

*Not yet written.* This section will cover:

- **6.1 Numeric to numeric** — Pearson, Spearman and Kendall correlations with an explanation of why they differ, a correlation matrix and scatter plots.
- **6.2 Categorical to categorical** — contingency tables and Cramér's V, plus categorical-to-numeric relationships via binning.
- **6.3 Graphs** — scatterplot, histogram, bar chart, box plot, violin plot, pie chart, pairplot and heatmap, each with titles, axis labels, a legend and written insights.

## 7. Index Analysis

*Not yet written.* This section will cover:

- Is the index unique? Is it time-based? Does the analysis above change over time? Is the data sorted?

## 8. Insights and the Data Story

*Not yet written.* This section will cover:

- At least three central insights, at least one bias or risk, possible failure points for engineering and statistical use, and what the analysis changed about my understanding of the domain.

## 9. Extensions

*Not yet written.* This section will cover:

- Time dependence and links to external knowledge about the period, feature engineering, hypothesis testing, and suggestions for further research.